In [3]:
import json
import pandas as pd
import os
from pathlib import Path

In [7]:
cd ..

c:\Users\Mohamed\Nextcloud\passim_BoH


# Data extraction

In [5]:
#input_dir = Path(r"04.CoMMa\parts")
input_dir = Path(r"C:\Users\yatsyk\deep-jsonl")
output_csv = Path(r"C:\Users\yatsyk\Documents\NextCloud_Synchro\passim_BoH\04.CoMMa\CoMMA_summary.csv")

In [6]:

wanted_fields = [
    "biblissima_id",
    "iiif_manifest",
    "notBefore",
    "notAfter",
    "shelfmark",
    "language_source",
    "language_normalized",
    "language_recategorized",
    "language_categorized",
]

rows = []

jsonl_files = sorted(input_dir.glob("*.jsonl"))

print(f"Found {len(jsonl_files)} JSONL files")

for jsonl_path in jsonl_files:

    print(f"Processing {jsonl_path.name}")

    with open(jsonl_path, "r", encoding="utf-8") as f:

        for line_num, line in enumerate(f, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except Exception as e:
                print(f"Error in {jsonl_path.name}, line {line_num}: {e}")
                continue

            files = record.get("files", [])

            pages = len(files)

            pages_with_text = 0
            lines_total = 0
            text_parts = []

            for page in files:

                page_has_text = False

                for zone in page.get("zones", []):

                    for text_line in zone.get("lines", []):

                        content = text_line.get("content", "")

                        if content and str(content).strip():

                            page_has_text = True
                            lines_total += 1
                            text_parts.append(str(content).strip())

                if page_has_text:
                    pages_with_text += 1

            tokens_from_lines = len(" ".join(text_parts).split())

            row = {field: record.get(field) for field in wanted_fields}

            row.update({
                "source_jsonl": jsonl_path.name,
                "jsonl_line_number": line_num,
                "pages": pages,
                "pages_with_text": pages_with_text,
                "lines_total": lines_total,
                "tokens": record.get("tokens"),
            })

            rows.append(row)

summary_df = pd.DataFrame(rows)

summary_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"Saved {len(summary_df):,} records")
print(f"Output: {output_csv}")

summary_df.head()

Found 33 JSONL files
Processing parts_0000.jsonl
Processing parts_0001.jsonl
Processing parts_0002.jsonl
Processing parts_0003.jsonl
Processing parts_0004.jsonl
Processing parts_0005.jsonl
Processing parts_0006.jsonl
Processing parts_0007.jsonl
Processing parts_0008.jsonl
Processing parts_0009.jsonl
Processing parts_0010.jsonl
Processing parts_0011.jsonl
Processing parts_0012.jsonl
Processing parts_0013.jsonl
Processing parts_0014.jsonl
Processing parts_0015.jsonl
Processing parts_0016.jsonl
Processing parts_0017.jsonl
Processing parts_0018.jsonl
Processing parts_0019.jsonl
Processing parts_0020.jsonl
Processing parts_0021.jsonl
Processing parts_0022.jsonl
Processing parts_0023.jsonl
Processing parts_0024.jsonl
Processing parts_0025.jsonl
Processing parts_0026.jsonl
Processing parts_0027.jsonl
Processing parts_0028.jsonl
Processing parts_0029.jsonl
Processing parts_0030.jsonl
Processing parts_0031.jsonl
Processing parts_0032.jsonl
Saved 32,763 records
Output: C:\Users\yatsyk\Documents\

,biblissima_id,iiif_manifest,notBefore,notAfter,shelfmark,language_source,language_normalized,language_recategorized,language_categorized,source_jsonl,jsonl_line_number,pages,pages_with_text,lines_total,tokens
0,https://data.biblissima.fr/entity/Q47972,https://gallica.bnf.fr/iiif/ark:/12148/btv1b90...,1401,1500,"Paris. Bibliothèque nationale de France, Dépar...",fre,[fre],[French],Truly French,parts_0000.jsonl,1,241,241,29773,161400
1,https://data.biblissima.fr/entity/Q206328,https://api.irht.cnrs.fr/ark:/63955/fce70txenq...,1001,1100,"Orléans. Médiathèque, MS 163","latin, latin, lat",[lat],[Latin],Massively Latin,parts_0000.jsonl,2,32,31,1030,5941
2,https://data.biblissima.fr/entity/Q243160,https://api.irht.cnrs.fr/ark:/63955/fkrsdc504s...,1427,1427,"Nancy. Bibliothèque municipale, Ms. 354",lat,[lat],[Latin],Massively bilingual,parts_0000.jsonl,3,2,2,93,281
3,https://data.biblissima.fr/entity/Q47353,https://gallica.bnf.fr/iiif/ark:/12148/btv1b84...,1401,1405,"Paris. Bibliothèque nationale de France, Dépar...",fre,[fre],[French],Truly French,parts_0000.jsonl,4,195,188,6596,34500
4,https://data.biblissima.fr/entity/Q48290,https://gallica.bnf.fr/iiif/ark:/12148/btv1b84...,1250,1275,"Paris. Bibliothèque nationale de France, Dépar...",fre,[fre],[French],Truly French,parts_0000.jsonl,5,586,563,56842,415500


# Data analysis

In [9]:
df_CoMMA = pd.read_csv("04.CoMMa\\CoMMA_summary.csv")
df_CoMMA

,biblissima_id,iiif_manifest,notBefore,notAfter,shelfmark,language_source,language_normalized,language_recategorized,language_categorized,source_jsonl,jsonl_line_number,pages,pages_with_text,lines_total,tokens
0,https://data.biblissima.fr/entity/Q47972,https://gallica.bnf.fr/iiif/ark:/12148/btv1b90...,1401,1500,"Paris. Bibliothèque nationale de France, Dépar...",fre,['fre'],['French'],Truly French,parts_0000.jsonl,1,241,241,29773,161400
1,https://data.biblissima.fr/entity/Q206328,https://api.irht.cnrs.fr/ark:/63955/fce70txenq...,1001,1100,"Orléans. Médiathèque, MS 163","latin, latin, lat",['lat'],['Latin'],Massively Latin,parts_0000.jsonl,2,32,31,1030,5941
2,https://data.biblissima.fr/entity/Q243160,https://api.irht.cnrs.fr/ark:/63955/fkrsdc504s...,1427,1427,"Nancy. Bibliothèque municipale, Ms. 354",lat,['lat'],['Latin'],Massively bilingual,parts_0000.jsonl,3,2,2,93,281
3,https://data.biblissima.fr/entity/Q47353,https://gallica.bnf.fr/iiif/ark:/12148/btv1b84...,1401,1405,"Paris. Bibliothèque nationale de France, Dépar...",fre,['fre'],['French'],Truly French,parts_0000.jsonl,4,195,188,6596,34500
4,https://data.biblissima.fr/entity/Q48290,https://gallica.bnf.fr/iiif/ark:/12148/btv1b84...,1250,1275,"Paris. Bibliothèque nationale de France, Dépar...",fre,['fre'],['French'],Truly French,parts_0000.jsonl,5,586,563,56842,415500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32758,https://data.biblissima.fr/entity/Q417603,https://api.irht.cnrs.fr/ark:/63955/fdmg0aekaj...,1401,1500,"Aix-en-Provence. Bibliothèque Méjanes, Ms. 1756",lat,['lat'],['Latin'],Massively Latin,parts_0032.jsonl,759,3,3,53,433
32759,https://data.biblissima.fr/entity/Q61485,https://gallica.bnf.fr/iiif/ark:/12148/btv1b90...,1301,1400,"Paris. Bibliothèque nationale de France, Dépar...",lat,['lat'],['Latin'],Truly Latin,parts_0032.jsonl,760,282,280,60654,651300
32760,https://data.biblissima.fr/entity/Q59985,https://gallica.bnf.fr/iiif/ark:/12148/btv1b90...,1101,1200,"Paris. Bibliothèque nationale de France, Dépar...",lat,['lat'],['Latin'],Ambiguous,parts_0032.jsonl,761,122,120,12594,193900
32761,https://data.biblissima.fr/entity/Q242438,https://gallica.bnf.fr/iiif/ark:/12148/btv1b10...,1201,1300,"Grenoble. Bibliothèque municipale, Ms.41 Rés.",lat,['lat'],['Latin'],Truly Latin,parts_0032.jsonl,762,421,414,28854,174600
